In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("OPENROUTER_API_KEY")

print("API key loaded:", api_key is not None)

API key loaded: True


In [3]:
from openai import OpenAI

client = OpenAI(
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1"
)

response = client.chat.completions.create(
    model="openrouter/free",
    messages=[
        {"role": "user", "content": "Explain what RAG is in one sentence."}
    ]
)

print(response.choices[0].message.content)

RAG (Retrieval-Augmented Generation) is a framework that enhances language‑model outputs by retrieving relevant external documents and incorporating their information into the generation process.


In [4]:
from src.pdf_loader import load_pdf
from src.chunker import create_chunks
from src.embeddings import generate_embeddings
from src.vector_store import create_vector_store
from src.retriever import retrieve

pdf_path = r"C:\Users\ASUS\OneDrive\Desktop\RAG-Document-Intelligence\data\sample.pdf"

pages = load_pdf(pdf_path)
chunks = create_chunks(pages)

texts = [chunk["text"] for chunk in chunks]
embeddings = generate_embeddings(texts)

index = create_vector_store(embeddings)

print("RAG pipeline ready")

C:\Users\ASUS\anaconda3\envs\rag-project\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3410.46it/s]


RAG pipeline ready


In [5]:
query = "What is the Real Time AI Voice Agent Interview Platform?"

query_embedding = generate_embeddings([query])

results = retrieve(
    index,
    query_embedding,
    chunks,
    k=3
)

for result in results:
    print("Page:", result["page"])
    print(result["text"])
    print("-" * 80)

Page: 1
 
 
A Project Report 
 
On 
 
REAL TIME AI VOICE AGENT INTERVIEW PLATFORM WITH 
GENERATIVE CONVERSATIONAL AI 
PROJECT ID: BT0364 
Submitted in partial fulfillment of the 
requirement for the award of the degree of 
BACHELOR OF TECHNOLOGY 
 
 
 
 
B-Tech 
Session 2025-26 
in 
Computer Science and Engineering 
By 
SUFIYAN HEBBAL: 22SCSE1012949 
SAQUIB HUSSAIN: 22SCSE1012964 
Under the guidance of  
Ms. ANKITA GUPTA 
Assistant Professor 
SCHOOL OF COMPUTING SCIENCE AND ENGINEERING DEPARTMENT OF 
COMPUTER SCIENCE AND ENGINEERING 
GALGOTIAS UNIVERSITY, GREATER NOIDA 
INDIA 
JUNE, 2026 

--------------------------------------------------------------------------------
Page: 45
46 
 
Table D: Comparison with Existing AI Interview Platforms 
 
Platform Type 
Interaction 
Mode 
Latency 
Cloud 
Dependency 
Key Limitation 
Text-Based 
(e.g., 
HireVue text)[6] 
Text Q&A 
N/A 
Yes 
No 
speech 
assessment 
Pre-recorded Video 
(e.g., 
HireVue 
video)[11] 
One-way 
video 
Async 
Yes 
No 
real-t

In [7]:
context = "\n\n".join(
    [
        f"[Page {result['page']}]\n{result['text']}"
        for result in results
    ]
)

prompt = f"""
Answer the question using only the provided context.

For every answer, mention the page numbers of the relevant sources.

Context:
{context}

Question:
{query}

Answer:
"""

response = client.chat.completions.create(
    model="openrouter/free",
    messages=[
        {"role": "user", "content": prompt}
    ]
)

answer = response.choices[0].message.content

print(answer)

The **Real Time AI Voice Agent Interview Platform** is a B.Tech project platform for conducting interviews using **real-time voice interaction** and **generative conversational AI**. It is described as an integrated real-time AI voice agent pipeline that works **on-device**, with **no cloud dependency**, and aims to assess spontaneous conversational skills during interviews.  

Relevant sources: the project title and purpose are stated on **Page 1**; its comparison as a real-time voice, on-device system is given on **Page 45**; and its goal of integrating and evaluating a real-time voice-agent pipeline is described on **Page 20**.
